# Note detection: preliminary CocoChorales, URMP and Bach10 experiment

Compare Attune's current pYIN+ → linear KernelCPD → score alignment and repeat-recovery pipeline with **Basic Pitch**, **CREPE Notes**, and **Tony/pYIN Vamp**. External methods keep their native complete transcription pipelines. This is an exploratory, score-informed application comparison, not an audio-only state-of-the-art claim.

Attune receives the supplied MIDI score part (shortest-note prior) and the same reference-F0-derived pitch range as `pitch.ipynb`; the other methods receive audio only. URMP's performed-note `Notes` annotations are used **only by the evaluator** (the separate frame-F0 annotations set the benchmark pitch range); Attune gets the separate `Sco_*.mid` part. Bach10 likewise uses original isolated recordings, supplied score MIDI parts, and separate `GTNotes.mat` performed-note references available only to the evaluator. The optional `attune-audio-only` ablation removes score access and uses fixed range/minimum duration. Attune includes production score alignment, robust timing refitting, and repeat recovery. No reference-based latency adjustment or boundary trimming is performed.

The existing notebooks are `../../notebooks/archive/benchmark_notes.ipynb` (historical PELT), `benchmarks.ipynb` (old combined entry point), and `sweeps/note.ipynb` (parameter selection). See [audit and competitor research](../../docs/archive/note-benchmark-audit-2026-09-23.md) before citing results.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'app.py').is_file()))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from IPython.display import display
import pandas as pd
import importlib
for name in ("NoteDetectorBase", "NoteCache", "competitors.Ruptures", "competitors.SlopeWindow",
             "competitors.Attune", "competitors.BasicPitch", "competitors.CrepeNotes", "competitors.Tony",
             "NoteBenchmarker", "NoteNotebook"):
    importlib.reload(importlib.import_module(f"benchmarks.modules.note.{name}"))
from benchmarks.modules.note.NoteNotebook import NotebookConfig
from benchmarks.modules.note.NoteNotebook import NoteNotebook
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook
PITCH_CONFIG = PitchNotebook.NotebookConfig()
CONFIG = NotebookConfig(seed=PITCH_CONFIG.seed, preliminary_tracks_per_instrument=PITCH_CONFIG.preliminary_tracks_per_instrument, workers=PITCH_CONFIG.workers, neural_workers=6, force_methods=('attune',), use_pitch_cache=PITCH_CONFIG.use_cache)
BENCH = NoteNotebook(CONFIG)
BENCH.show_configuration()


,setting,value
0,workers,9
1,neural_workers,6
2,preliminary_tracks_per_instrument,2
3,seed,0
4,max_tracks,None
5,methods,"(attune, basic-pitch, crepe-notes, tony)"
6,onset_tolerance,0.05
7,pitch_tolerance,50.0
8,offset_ratio,0.2
9,offset_min_tolerance,0.05


,setting,value
0,workers,9
1,neural_workers,6
2,preliminary_tracks_per_instrument,2
3,seed,0
4,max_tracks,None
5,methods,"(attune, basic-pitch, crepe-notes, tony)"
6,onset_tolerance,0.05
7,pitch_tolerance,50.0
8,offset_ratio,0.2
9,offset_min_tolerance,0.05


## Historical evidence audit

Read the original per-track CSVs, not `note_benchmarks.csv` (currently just two one-track experimental rows). No old result is overwritten. The saved linear KernelCPD and external baseline rows share 2,486 Coco stems, but old processing settings and code provenance differ from the shipped pipeline.

In [2]:
historical_rows, historical_summary = BENCH.historical_audit()
display(historical_summary)
if not historical_rows.empty:
    paired_old = historical_rows.pivot(
        index="Track ID", columns="saved_method", values="F-measure"
    )
    display(
        paired_old[["kernelcpd-linear", "basic-pitch", "crepe-notes", "tony"]]
        .dropna()
        .agg(["count", "mean"])
    )


,size,count,mean
saved_method,,,
pelt-l2,2287,2287,0.968360
kernelcpd-cosine,2486,2486,0.964407
kernelcpd-linear,2486,2486,0.964407
kernelcpd-gaussian,2486,2486,0.964179
pelt-l1,2486,2486,0.959130
pelt-rbf,2486,2486,0.956329
crepe-notes,2486,2486,0.938703
basic-pitch,2486,2486,0.845363
tony,2486,2486,0.833381


saved_method,kernelcpd-linear,basic-pitch,crepe-notes,tony
count,2486.000000,2486.000000,2486.000000,2486.000000
mean,0.964407,0.845363,0.938703,0.833381


## Protocol and CPU timing

Seed 0 selects two stems per available instrument, using the existing pitch dataset samplers. Coco uses the test split, original isolated stem audio and stem MIDI. URMP uses isolated `AuSep` recordings and official `Notes` files: onset seconds, frequency Hz, duration seconds. Bach10 uses the same eight original recordings as the pitch preliminary (two each of violin, clarinet, saxophone and bassoon). Its `GTNotes.mat` references supply performed boundaries on the 10 ms grid: MATLAB frame 1 is centered at 23 ms, and first/last frame centers define onset/offset. Each note is assigned the median fractional MIDI pitch of its annotated contour, converted to Hz without rounding to a score pitch. Supplied score MIDI parts condition Attune separately; interpolated `.txt` score-alignment times are not used as note ground truth. Full recordings are scored. Selection manifests are saved before execution; missing selected inputs fail loudly.

Primary quality: pooled-note (micro) precision/recall/F1 within each dataset with **50 ms onset and 50 cents pitch** tolerance, offsets ignored. Secondary: the same metric requiring offsets within **max(50 ms, 20% of reference duration)**. Empty predictions score zero; dependency/runtime failures remain errors, never zeros. Full polyphonic Basic Pitch predictions are retained, including simultaneous onsets.

Jobs run in fresh spawned processes, normally CPU count minus one workers. Heavy neural methods use up to six workers to bound memory. Every numerical runtime gets one thread; GPU execution is disabled. Results include:

| Column | Measured scope |
|---|---|
| `cpu_seconds` | Full algorithm CPU: frontend + note stage; a reused frontend retains its original measured CPU cost. Tony reports its complete call because stages are unavailable |
| `execution_cpu_seconds` | CPU actually spent in this invocation, including imports/setup/cache loading; a cache hit makes this smaller without making the algorithm appear free |
| `frontend_cpu_seconds` | Attune pitch detection+smoothing; Basic Pitch neural inference; CREPE tracking |
| `segmentation_cpu_seconds` | Attune score-span/minimum-length preparation + production KernelCPD; Basic Pitch native note creation; CREPE Notes native post-processing including amplitude analysis and MIDI writing |
| `wall_seconds` | Observed invocation elapsed time; cache- and contention-dependent diagnostic |
| `audio_per_cpu` | Total audio duration divided by total full-call CPU seconds |

CPU time is `process_time` plus reaped child user/system CPU, so Tony's Sonic Annotator work is counted. Tony exposes no separate segmentation timer: stage columns are missing, not zero. Stage boundaries are not identical algorithms: Basic Pitch has already inferred onset/note activations before note creation. Compare stage times with that qualification. CPU timing avoids charging scheduling waits, though contention can still affect CPU efficiency. Stage timings include different native loading/model-initialization costs: Attune follows the pitch notebook’s detector+smoother timing, while the external APIs load audio/models within their frontend. Neither aggregate CPU nor invocation time is steady-state interactive latency.

Attune reuses `pitch.ipynb`’s full smoothed `PitchData` cache, at the same sample rate, range, seed and selected stem IDs. Reuse requires the current cache version, production smoother version and process-CPU timing; misses use the current production frontend and update the shared cache. Old wall-clock or incompatible caches are not accepted. These legacy cache files do not contain full source/config fingerprints, so the compatibility guarantee is their version contract, not proof of arbitrary older settings; set `use_pitch_cache=False` to force a fresh frontend. Successful note checkpoints are fingerprinted and resumable; errors are retried. `force_methods=("attune",)` (the notebook default) repeats only Attune note jobs; competitors reuse compatible completed checkpoints. Set `force_methods=()` to resume Attune too. `force=True` repeats all note jobs but retains compatible pitch caches unless disabled. Each run saves input/source hashes, versions, config, selection, predictions, per-job logs, raw rows, summaries and paired comparisons under `results/note/notebook_runs/`. The existing CREPE pitch cache lacks raw confidence and Basic Pitch’s cache lacks note/onset activations, so neither is a valid native note-decoder input. Their full predictions are reused through successful note-run checkpoints instead.

Live progress uses the exact `PitchBenchmarker.Progress` compact renderer from `pitch.ipynb`: one updating `[   n/total] method: track_id` line across all selected method/track pairs, with the same track IDs. Cached and newly computed tracks each advance this counter once; fully cached runs use the same display and reach the same total. Errors are listed after the live line finishes; full Python/native runtime output stays in each job’s log.


CREPE Notes retains its native CNN onset splitting. Its adapter resolves installed FFmpeg/ffprobe tools even when the notebook PATH omits Homebrew; this fixes the earlier decoder failures on CocoChorales and URMP. Missing decoder tools now fail before expensive pitch inference. Reports retain full provenance in separate directories. Completed jobs are reused across directories using per-recording input hashes, inference/scoring code, relevant dependencies, and scoring settings; worker counts, method selection, Git HEAD, and unrelated benchmark edits do not invalidate them. Compatible checkpoints from the stopped three-worker run are recognized automatically. Each returned job is atomically checkpointed before the status counter advances. Interruptions preserve completed jobs; set `force_methods=()` to resume only unfinished or failed jobs. `force=True` explicitly bypasses result reuse. CREPE intermediate pitch estimates are not separately checkpointed, so an unfinished recording restarts in full.


Attune now includes production score alignment, robust timing refitting, and repeat-note recovery on every dataset. Recovered notes are scored against the unchanged references. `refinement_cpu_seconds` records this additional stage and is included in `cpu_seconds`. The optional `attune-audio-only` diagnostic remains score-free and excludes recovery.


## Preliminary CocoChorales

This sample may overlap earlier note sweeps; it is a development check. Do not treat the test-split name as proof it was never used for model selection.

In [3]:
preliminary_coco = BENCH.run_preliminary_coco()
display(BENCH.summarize(preliminary_coco))


Running: preliminary_coco (78 cached, 26 pending)
[ 104/104] tony: test__woodwind_track229675__2_oboe            
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_coco_dae5b60cc68820df


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,f1_offset,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,coco,attune,26,26,0,1,True,pooled_note_counts,571,38,...,0.935961,0.939044,0.937500,2.506542,1.428654,1.034385,2.457583,0.033437,0.015522,8.283003
1,coco,basic-pitch,26,26,0,1,True,pooled_note_counts,576,365,...,0.523911,0.812191,0.636951,4.446164,9.792395,9.358705,4.434179,0.011985,NaN,4.669574
2,coco,crepe-notes,26,26,0,1,True,pooled_note_counts,590,81,...,0.870343,0.962109,0.913928,109.127559,116.558675,113.612608,108.455401,0.672158,NaN,0.190252
3,coco,tony,26,26,0,1,True,pooled_note_counts,347,200,...,0.581353,0.523888,0.551127,2.125452,2.840571,2.125452,NaN,NaN,NaN,9.768129


## Preliminary URMP

Current production pitch settings were selected using the two-per-instrument URMP development sample. These results are not independent held-out evidence. Attune's score-informed row and the audio-only external rows have different input privileges.

In [4]:
preliminary_urmp = BENCH.run_preliminary_urmp()
display(BENCH.summarize(preliminary_urmp))


Running: preliminary_urmp (78 cached, 26 pending)
[ 104/104] tony: 43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_5_tba_43_Chorale          
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_urmp_04759c915b4ac896


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,f1_offset,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,urmp,attune,26,26,0,1,True,pooled_note_counts,3739,431,...,0.861871,0.855103,0.858474,13.247367,3.480541,2.879777,12.018568,0.107889,1.12091,6.874222
1,urmp,basic-pitch,26,26,0,1,True,pooled_note_counts,3492,1422,...,0.583639,0.682370,0.629154,7.465826,12.275627,11.462691,7.345444,0.120382,NaN,12.197622
2,urmp,crepe-notes,26,26,0,1,True,pooled_note_counts,3662,1140,...,0.682007,0.779205,0.727374,478.449774,493.185156,483.124308,475.314229,3.135545,NaN,0.190334
3,urmp,tony,26,26,0,1,True,pooled_note_counts,2291,290,...,0.811701,0.498453,0.617630,3.880640,4.370081,3.880640,NaN,NaN,NaN,23.466577


## Preliminary Bach10 — original recordings

The balanced selection matches `pitch.ipynb`, allowing Attune to reuse compatible production pitch caches. The four instrument parts are evaluated separately; ensemble mixes are excluded. Frame references were generated with YIN and manual corrections, then grouped into notes manually. See the [release description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

This uses the same spawned worker pool, neural worker limit, single updating status line, metrics, and report format as the CocoChorales and URMP cells. The default run has 32 jobs (8 recordings × 4 methods). Results and manifests are saved under a separate `preliminary_bach10-original_*` directory. The sample is a development comparison, not independent held-out evidence.


In [5]:
preliminary_bach10 = BENCH.run_preliminary_bach10()
display(BENCH.summarize(preliminary_bach10))


Running: preliminary_bach10-original (24 cached, 8 pending)
[  32/32] tony: 10-NunBitten-saxphone               
Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/note/notebook_runs/preliminary_bach10-original_b55aaf1c54d5d5a9


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,f1_offset,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,bach10-original,attune,8,8,0,1,True,pooled_note_counts,340,58,...,0.801508,0.778049,0.789604,5.524780,1.482656,1.411678,5.066717,0.043980,0.414083,6.328727
1,bach10-original,basic-pitch,8,8,0,1,True,pooled_note_counts,367,133,...,0.690000,0.841463,0.758242,5.245647,11.025698,9.419733,5.218829,0.026819,NaN,6.665493
2,bach10-original,crepe-notes,8,8,0,1,True,pooled_note_counts,365,67,...,0.766204,0.807317,0.786223,184.403210,193.813907,189.063659,183.329956,1.073254,NaN,0.189611
3,bach10-original,tony,8,8,0,1,True,pooled_note_counts,349,41,...,0.835897,0.795122,0.815000,2.595808,2.938247,2.595808,NaN,NaN,NaN,13.469729


## Paired quality, coverage and instrument breakdown

Differences are pooled-note micro-F1 for Attune minus baseline on successful matching instrument tracks, reported separately per corpus. Exploratory 95% percentile intervals resample whole pieces/tracks (2,000 draws), keeping sibling stems together. They do not establish equivalence: a “comparable” claim needs a prespecified non-inferiority margin and independent evaluation. `complete=False` flags missing pairs. Do not rank methods using unequal coverage without reporting it.

In [6]:
rows = pd.concat(
    [preliminary_coco, preliminary_urmp, preliminary_bach10], ignore_index=True
)
display(BENCH.summarize(rows))
display(BENCH.paired_comparisons(rows))
display(rows.loc[rows.status != "ok", ["dataset", "method", "track_id", "error"]])
good = rows[rows.status == "ok"]
if not good.empty:
    display(
        pd.concat(
            [
                BENCH.summarize(group).assign(instrument=instrument)
                for (_, instrument), group in good.groupby(["dataset", "instrument"])
            ],
            ignore_index=True,
        )[
            [
                "dataset",
                "instrument",
                "method",
                "succeeded",
                "tp",
                "fp",
                "fn",
                "f1",
                "f1_offset",
            ]
        ]
    )


,dataset,method,attempted,succeeded,failed,datasets_scored,complete,aggregation,tp,fp,...,precision_offset,recall_offset,f1_offset,cpu_seconds,wall_seconds,execution_cpu_seconds,frontend_cpu_seconds,segmentation_cpu_seconds,refinement_cpu_seconds,audio_per_cpu
0,bach10-original,attune,8,8,0,1,True,pooled_note_counts,340,58,...,0.801508,0.778049,0.789604,5.524780,1.482656,1.411678,5.066717,0.043980,0.414083,6.328727
1,bach10-original,basic-pitch,8,8,0,1,True,pooled_note_counts,367,133,...,0.690000,0.841463,0.758242,5.245647,11.025698,9.419733,5.218829,0.026819,NaN,6.665493
2,bach10-original,crepe-notes,8,8,0,1,True,pooled_note_counts,365,67,...,0.766204,0.807317,0.786223,184.403210,193.813907,189.063659,183.329956,1.073254,NaN,0.189611
3,bach10-original,tony,8,8,0,1,True,pooled_note_counts,349,41,...,0.835897,0.795122,0.815000,2.595808,2.938247,2.595808,NaN,NaN,NaN,13.469729
4,coco,attune,26,26,0,1,True,pooled_note_counts,571,38,...,0.935961,0.939044,0.937500,2.506542,1.428654,1.034385,2.457583,0.033437,0.015522,8.283003
5,coco,basic-pitch,26,26,0,1,True,pooled_note_counts,576,365,...,0.523911,0.812191,0.636951,4.446164,9.792395,9.358705,4.434179,0.011985,NaN,4.669574
6,coco,crepe-notes,26,26,0,1,True,pooled_note_counts,590,81,...,0.870343,0.962109,0.913928,109.127559,116.558675,113.612608,108.455401,0.672158,NaN,0.190252
7,coco,tony,26,26,0,1,True,pooled_note_counts,347,200,...,0.581353,0.523888,0.551127,2.125452,2.840571,2.125452,NaN,NaN,NaN,9.768129
8,urmp,attune,26,26,0,1,True,pooled_note_counts,3739,431,...,0.861871,0.855103,0.858474,13.247367,3.480541,2.879777,12.018568,0.107889,1.120910,6.874222
9,urmp,basic-pitch,26,26,0,1,True,pooled_note_counts,3492,1422,...,0.583639,0.682370,0.629154,7.465826,12.275627,11.462691,7.345444,0.120382,NaN,12.197622


,dataset,baseline,metric,aggregation,paired_tracks,groups,delta,ci_low,ci_high,complete
0,bach10-original,basic-pitch,f1,pooled_note_counts,8,8,0.034991,-0.082420,0.151916,True
1,bach10-original,crepe-notes,f1,pooled_note_counts,8,8,-0.025399,-0.074118,0.017470,True
2,bach10-original,tony,f1,pooled_note_counts,8,8,-0.030916,-0.064480,0.002277,True
3,bach10-original,basic-pitch,f1_offset,pooled_note_counts,8,8,0.031362,-0.105373,0.148354,True
4,bach10-original,crepe-notes,f1_offset,pooled_note_counts,8,8,0.003381,-0.064573,0.063739,True
5,bach10-original,tony,f1_offset,pooled_note_counts,8,8,-0.025396,-0.073452,0.018171,True
6,coco,basic-pitch,f1,pooled_note_counts,26,26,0.194959,0.157077,0.230097,True
7,coco,crepe-notes,f1,pooled_note_counts,26,26,0.015827,-0.022003,0.052301,True
8,coco,tony,f1,pooled_note_counts,26,26,0.337758,0.266798,0.420837,True
9,coco,basic-pitch,f1_offset,pooled_note_counts,26,26,0.300549,0.243725,0.365351,True


,dataset,method,track_id,error


,dataset,instrument,method,succeeded,tp,fp,fn,f1,f1_offset
0,bach10-original,bassoon,attune,2,87,3,6,0.950820,0.874317
1,bach10-original,bassoon,basic-pitch,2,85,73,8,0.677291,0.645418
2,bach10-original,bassoon,crepe-notes,2,90,17,3,0.900000,0.800000
3,bach10-original,bassoon,tony,2,88,6,5,0.941176,0.909091
4,bach10-original,clarinet,attune,2,90,21,27,0.789474,0.736842
...,...,...,...,...,...,...,...,...,...
115,urmp,viola,tony,2,210,33,141,0.707071,0.629630
116,urmp,violin,attune,2,467,8,12,0.979036,0.966457
117,urmp,violin,basic-pitch,2,436,79,43,0.877264,0.659960
118,urmp,violin,crepe-notes,2,455,65,24,0.910911,0.796797


## Competitor follow-up and claim boundary

The historical results include Basic Pitch, CREPE Notes and Tony plus PELT (L1/L2/RBF), KernelCPD (linear/Gaussian/cosine), bottom-up and window segmentation; a one-track neural-boundary experiment is not a meaningful competitor benchmark. Current configuration also lists oracle-count Dynp and slope-aware windows. MT3 is only an unimplemented placeholder.

Notable additions: [MT3](https://github.com/magenta/mt3) and [YourMT3+](https://arxiv.org/abs/2407.04822) for stronger multi-instrument transcription, plus the newly released [MuScriptor (2026)](https://arxiv.org/abs/2607.08168); audit checkpoint training overlap and use appropriate URMP held-out splits before evaluating them. [Melodia → notes](https://github.com/justinsalamon/audio_to_midi_melodia) is a useful classical sanity baseline, but its authors describe the segmenter as deliberately simple. Tony is already represented and should remain in the default experiment. Piano-only transcription systems are not direct replacements for this varied-instrument suite.

The historical descriptive result is: “In the saved CocoChorales benchmark, the linear KernelCPD configuration achieved higher mean onset-and-pitch F1 than Basic Pitch and CREPE Notes on the same 2,486 stems.” A present-tense claim about current Attune requires the fresh results, explicit input/metric conditions, and preferably an untouched evaluation set. Neither notebook establishes performance on mixtures, live microphone recordings, or unseen instruments.

## Overall metrics across datasets

Pool true positives, false positives, and false negatives across all successful instrument tracks in all three datasets, then compute micro precision, recall, and F1. Matching is performed separately within each recording. Onset-only and offset-aware matches are pooled separately. Failed recordings remain reported as failures, not zero scores; `complete` flags incomplete coverage. CPU time remains an average per successful recording, and `audio_per_cpu` divides total audio duration by total algorithm CPU time.


In [7]:
overall_rows = pd.concat(
    [preliminary_coco, preliminary_urmp, preliminary_bach10], ignore_index=True
)
overall_summary = BENCH.summarize(overall_rows, by_dataset=False).set_index("method")
quality_metrics = [
    "precision",
    "recall",
    "f1",
    "precision_offset",
    "recall_offset",
    "f1_offset",
]
overall_summary[quality_metrics] *= 100
overall_summary = (
    overall_summary[
        [
            "attempted",
            "succeeded",
            "failed",
            "datasets_scored",
            "complete",
            "tp",
            "fp",
            "fn",
            *quality_metrics[:3],
            "tp_offset",
            "fp_offset",
            "fn_offset",
            *quality_metrics[3:],
            "cpu_seconds",
            "audio_per_cpu",
        ]
    ]
    .rename(
        columns={
            "precision": "Micro precision (%)",
            "recall": "Micro recall (%)",
            "f1": "Micro F1 (%)",
            "precision_offset": "Offset micro precision (%)",
            "recall_offset": "Offset micro recall (%)",
            "f1_offset": "Offset micro F1 (%)",
            "cpu_seconds": "CPU seconds / recording",
        }
    )
    .sort_values("Micro F1 (%)", ascending=False)
)
display(
    overall_summary.style.format(
        {
            column: "{:.2f}"
            for column in overall_summary.columns
            if "(%)" in column or column in ("CPU seconds / recording", "audio_per_cpu")
        },
        na_rep="—",
    )
)


,attempted,succeeded,failed,datasets_scored,complete,tp,fp,fn,Micro precision (%),Micro recall (%),Micro F1 (%),tp_offset,fp_offset,fn_offset,Offset micro precision (%),Offset micro recall (%),Offset micro F1 (%),CPU seconds / recording,audio_per_cpu
method,,,,,,,,,,,,,,,,,,,
attune,60,60,0,3,True,4650,527,570,89.82,89.08,89.45,4483,694,737,86.59,85.88,86.24,7.56,7.02
crepe-notes,60,60,0,3,True,4617,1288,603,78.19,88.45,83.00,4190,1715,1030,70.96,80.27,75.33,279.20,0.19
basic-pitch,60,60,0,3,True,4435,1920,785,69.79,84.96,76.63,3706,2649,1514,58.32,71.00,64.03,5.86,9.06
tony,60,60,0,3,True,2987,531,2233,84.91,57.22,68.37,2739,779,2481,77.86,52.47,62.69,2.95,18.01


## Paired significance of overall note F1

Recompute two-sided paired permutation tests from `overall_rows` above, without rerunning inference. The default family is Attune versus Basic Pitch, CREPE Notes, and Tony on **onset/pitch micro-F1**, with Holm correction across the three comparisons at 0.05. Positive differences favor Attune. Adding `"f1_offset"` to `SIGNIFICANCE_METRICS` includes those comparisons in the same Holm family.

All comparisons use the common successful track intersection. Swaps keep whole source groups together, conservatively merging repeated URMP titles, and recompute pooled F1 from swapped TP/FP/FN counts. Small designs enumerate every swap; larger designs use 9,999 random swaps with a plus-one correction. Coverage reports excluded tracks; failed runs are never assigned zero scores. `SOURCE_GROUPS` can link known shared compositions across datasets. The same cell supports larger runs by consuming their rows from the preceding overall-results cell; significance does not establish held-out generalization or superiority for every instrument.


In [ ]:
import benchmarks.modules.note.NoteBenchmarker as note_significance_module
importlib.reload(note_significance_module)
SIGNIFICANCE_METHODS = ('attune', 'basic-pitch', 'crepe-notes', 'tony')
SIGNIFICANCE_METRICS = ('f1',)
SIGNIFICANCE_RESAMPLES = 9999
SIGNIFICANCE_ALPHA = 0.05
SOURCE_GROUPS = {}
note_significance, significance_coverage = note_significance_module.NoteEvaluation.paired_significance(overall_rows, methods=SIGNIFICANCE_METHODS, metrics=SIGNIFICANCE_METRICS, n_resamples=SIGNIFICANCE_RESAMPLES, seed=CONFIG.seed, alpha=SIGNIFICANCE_ALPHA, source_groups=SOURCE_GROUPS)
display(significance_coverage)
display(note_significance.style.format({'attune_f1': '{:.2f}', 'competitor_f1': '{:.2f}', 'difference_pp': '{:+.2f}', 'p_value': '{:.4g}', 'p_adj': '{:.4g}', 'alpha': '{:.2f}'}))
print('F1 columns are percentages; difference_pp is Attune minus competitor.')
print('significant tests p_adj <= alpha (two-sided); inspect difference_pp for direction.')
